# L8.2 — Tool Calling: schemas, parsing, errors, safety

Hands-on notebook for the lesson [`8-2-tool-calling.mdx`](../../llm-quest-theory/level-8/8-2-tool-calling.mdx).

> **Learning objectives**
> - Declare tools as `pydantic v2` schemas and auto-derive the OpenAI-style JSON description.
> - Extract tool calls from a *messy* LLM output — tolerate JSON wrapped in prose, single quotes, and malformed fields.
> - Validate arguments against the schema, return a clean error to the LLM, and **retry**.
> - Add a safety gate that requires human confirmation before calling destructive tools.
> - Demonstrate **parallel** tool calls and a simple observability log.

## Connection to the theory
Covers **§1–§11** of the source `.mdx`. All mechanisms run locally — the "LLM" emits pre-recorded outputs so the notebook is deterministic. The parser, validator, and safety gate are the pieces that carry over verbatim to a real model.

In [1]:
# ---- Setup ----
import os, re, json, time, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

SEED = 42

## 1. Tool schemas with pydantic
Each tool has a `BaseModel` for its arguments. Pydantic gives us JSON schema generation *and* argument validation for free.

In [2]:
class SearchProductsArgs(BaseModel):
    query: str = Field(..., description="Free-text search over the product catalog")
    limit: int = Field(10, ge=1, le=50, description="Maximum results, 1-50")

class SendEmailArgs(BaseModel):
    to: str      = Field(..., description="Recipient email address")
    subject: str = Field(..., min_length=1, description="Email subject line")
    body: str    = Field(..., description="Email body (plain text)")

class DeleteOrderArgs(BaseModel):
    order_id: int         = Field(..., ge=1)
    reason:   str         = Field(..., min_length=3)
    confirm:  Literal[True] = Field(..., description="Must be explicitly True")

def tool_schema(model_cls, name, description):
    schema = model_cls.model_json_schema()
    return {
        "name": name,
        "description": description,
        "parameters": schema,
    }

TOOL_SCHEMAS = [
    tool_schema(SearchProductsArgs, "search_products",
                "Search the product catalog. Use for price / availability questions. Returns up to 50 rows."),
    tool_schema(SendEmailArgs, "send_email",
                "Send a non-destructive email to a verified address. Never use for password resets."),
    tool_schema(DeleteOrderArgs, "delete_order",
                "DESTRUCTIVE. Delete an order record. Requires confirm=true and human review."),
]
print(json.dumps(TOOL_SCHEMAS[0], indent=2))

{
  "name": "search_products",
  "description": "Search the product catalog. Use for price / availability questions. Returns up to 50 rows.",
  "parameters": {
    "properties": {
      "query": {
        "description": "Free-text search over the product catalog",
        "title": "Query",
        "type": "string"
      },
      "limit": {
        "default": 10,
        "description": "Maximum results, 1-50",
        "maximum": 50,
        "minimum": 1,
        "title": "Limit",
        "type": "integer"
      }
    },
    "required": [
      "query"
    ],
    "title": "SearchProductsArgs",
    "type": "object"
  }
}


## 2. The underlying tool implementations
The schemas are metadata only — real behavior lives here. We use a mock product database.

In [3]:
CATALOG = [
    {"id":  1, "name": "Mechanical Keyboard", "price": 90,   "stock": 14},
    {"id":  2, "name": "USB Mouse",           "price": 20,   "stock": 31},
    {"id":  3, "name": "4K Monitor",          "price": 320,  "stock": 5 },
    {"id":  4, "name": "Webcam HD",           "price": 45,   "stock": 0 },
    {"id":  5, "name": "Gaming Laptop",       "price": 1800, "stock": 2 },
]
EMAIL_LOG, DELETED = [], []

def search_products(query: str, limit: int = 10):
    q = query.lower()
    hits = [p for p in CATALOG if q in p["name"].lower()]
    return {"count": len(hits), "results": hits[:limit]}

def send_email(to: str, subject: str, body: str):
    EMAIL_LOG.append({"to": to, "subject": subject, "body": body})
    return {"status": "sent", "message_id": f"m{len(EMAIL_LOG):04d}"}

def delete_order(order_id: int, reason: str, confirm: bool):
    DELETED.append({"order_id": order_id, "reason": reason})
    return {"status": "deleted", "order_id": order_id}

IMPLEMENTATIONS = {
    "search_products": search_products,
    "send_email":      send_email,
    "delete_order":    delete_order,
}
ARG_MODELS = {
    "search_products": SearchProductsArgs,
    "send_email":      SendEmailArgs,
    "delete_order":    DeleteOrderArgs,
}

## 3. A tolerant JSON parser
A weaker LLM may return JSON wrapped in prose ("Sure! `{...}`") or use single quotes. Our parser extracts and loads it defensively.

In [4]:
def extract_json(text):
    """Return the first JSON object found in `text`, or None."""
    # 1. Try to find a fenced ```json block
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    candidate = fence.group(1) if fence else None
    # 2. Otherwise slice between the first '{' and the last '}'
    if candidate is None:
        lo, hi = text.find("{"), text.rfind("}")
        if lo == -1 or hi == -1 or hi <= lo:
            return None
        candidate = text[lo : hi + 1]
    # 3. Try strict JSON, then lenient (single quotes -> double)
    for attempt in (candidate, candidate.replace("'", '"')):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            continue
    return None

for sample in [
    '{"name": "search_products", "args": {"query": "laptop"}}',
    "Sure! Here is the call: ```json\n{\"name\": \"send_email\", \"args\": {}}\n```",
    "{'name': 'search_products', 'args': {'query': 'mouse', 'limit': 3}}",
    "I can't produce JSON, sorry.",
]:
    print(f"  parse ->", extract_json(sample))

  parse -> {'name': 'search_products', 'args': {'query': 'laptop'}}
  parse -> {'name': 'send_email', 'args': {}}
  parse -> {'name': 'search_products', 'args': {'query': 'mouse', 'limit': 3}}
  parse -> None


## 4. Validator + dispatcher
Given a parsed `{name, args}` blob, validate args against the matching schema, then call the tool and return the structured result (or a typed error).

In [5]:
class ToolError(Exception): pass

def dispatch(call):
    if not isinstance(call, dict) or "name" not in call or "args" not in call:
        return {"error": "bad_call_shape", "msg": "expected {name, args}"}
    name, args = call["name"], call["args"] or {}
    if name not in IMPLEMENTATIONS:
        return {"error": "unknown_tool", "msg": f"no such tool: {name!r}"}
    Model = ARG_MODELS[name]
    try:
        validated = Model(**args)
    except ValidationError as e:
        return {"error": "validation", "msg": str(e)}
    try:
        return IMPLEMENTATIONS[name](**validated.model_dump())
    except Exception as e:
        return {"error": "tool_failed", "msg": f"{type(e).__name__}: {e}"}

print(dispatch({"name": "search_products", "args": {"query": "monitor"}}))
print(dispatch({"name": "search_products", "args": {"query": "x", "limit": 500}}))
print(dispatch({"name": "no_such_tool",    "args": {}}))
print(dispatch("not a dict"))

{'count': 1, 'results': [{'id': 3, 'name': '4K Monitor', 'price': 320, 'stock': 5}]}
{'error': 'validation', 'msg': '1 validation error for SearchProductsArgs\nlimit\n  Input should be less than or equal to 50 [type=less_than_equal, input_value=500, input_type=int]\n    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal'}
{'error': 'unknown_tool', 'msg': "no such tool: 'no_such_tool'"}
{'error': 'bad_call_shape', 'msg': 'expected {name, args}'}


## 5. Simulated LLM outputs
Instead of actually calling a model, we hand-craft outputs that span clean, messy, and malformed cases.

In [6]:
FAKE_OUTPUTS = [
    # Clean
    '{"name": "search_products", "args": {"query": "keyboard", "limit": 2}}',
    # Wrapped in prose + fenced
    "Absolutely! Here it is:\n```json\n{\"name\": \"send_email\", \"args\": {\"to\":\"a@x.com\",\"subject\":\"Hi\",\"body\":\"Hello\"}}\n```\n",
    # Single quotes
    "{'name': 'search_products', 'args': {'query': 'mouse'}}",
    # Malformed JSON
    "search_products: query=laptop limit=5",
    # Argument limit too large -> validation error
    '{"name": "search_products", "args": {"query": "x", "limit": 999}}',
    # Destructive call without confirmation
    '{"name": "delete_order", "args": {"order_id": 42, "reason": "fraud"}}',
    # Destructive call with explicit confirmation
    '{"name": "delete_order", "args": {"order_id": 42, "reason": "fraud", "confirm": true}}',
]

for i, text in enumerate(FAKE_OUTPUTS):
    parsed = extract_json(text)
    print(f"[{i}] parsed={parsed}")

[0] parsed={'name': 'search_products', 'args': {'query': 'keyboard', 'limit': 2}}
[1] parsed={'name': 'send_email', 'args': {'to': 'a@x.com', 'subject': 'Hi', 'body': 'Hello'}}
[2] parsed={'name': 'search_products', 'args': {'query': 'mouse'}}
[3] parsed=None
[4] parsed={'name': 'search_products', 'args': {'query': 'x', 'limit': 999}}
[5] parsed={'name': 'delete_order', 'args': {'order_id': 42, 'reason': 'fraud'}}
[6] parsed={'name': 'delete_order', 'args': {'order_id': 42, 'reason': 'fraud', 'confirm': True}}


## 6. Safety gate — a human must confirm destructive actions
Even when the LLM requests a destructive tool, our dispatcher should refuse unless an **external authorisation token** is present. That token would come from a human click in a real product.

In [7]:
DESTRUCTIVE = {"delete_order"}

def dispatch_safe(call, human_auth_token=None):
    if isinstance(call, dict) and call.get("name") in DESTRUCTIVE and human_auth_token != "APPROVED":
        return {"error": "needs_human_approval",
                "msg": f"destructive tool {call.get('name')!r} needs APPROVED token"}
    return dispatch(call)

# Attempts without token are refused
print(dispatch_safe({"name": "delete_order",
                     "args": {"order_id": 42, "reason": "fraud", "confirm": True}}))
# With approval, it goes through
print(dispatch_safe({"name": "delete_order",
                     "args": {"order_id": 42, "reason": "fraud", "confirm": True}},
                    human_auth_token="APPROVED"))
print("log:", DELETED)

{'error': 'needs_human_approval', 'msg': "destructive tool 'delete_order' needs APPROVED token"}
{'status': 'deleted', 'order_id': 42}
log: [{'order_id': 42, 'reason': 'fraud'}]


## 7. Retry loop — tell the LLM what went wrong
When dispatch returns an error, the agent would normally loop: feed the error back, get a new tool call, retry. We simulate that with a fake regenerator that "fixes" the arguments.

In [8]:
def regenerate(last_call, error_msg):
    """Mock the LLM: read the error and propose a patch."""
    # Our heuristic just repairs the two cases in the suite.
    if last_call and last_call.get("name") == "search_products":
        args = dict(last_call.get("args", {}))
        if args.get("limit", 0) > 50:
            args["limit"] = 10
            return {"name": "search_products", "args": args}
    return None   # give up

def run_with_retry(call, max_retries=2, auth=None):
    for attempt in range(max_retries + 1):
        result = dispatch_safe(call, human_auth_token=auth)
        if "error" not in result:
            return {"attempts": attempt + 1, "result": result}
        patch = regenerate(call, result["msg"])
        if patch is None:
            return {"attempts": attempt + 1, "error": result}
        call = patch
    return {"attempts": max_retries + 1, "error": "exhausted"}

bad_call = {"name": "search_products", "args": {"query": "laptop", "limit": 999}}
print(json.dumps(run_with_retry(bad_call), indent=2))

{
  "attempts": 2,
  "result": {
    "count": 1,
    "results": [
      {
        "id": 5,
        "name": "Gaming Laptop",
        "price": 1800,
        "stock": 2
      }
    ]
  }
}


## 8. Parallel tool calls
When the LLM emits a list of independent tool calls, run them concurrently. On CPU we simulate concurrency with a thread pool — the key point is the call-list shape, not the speedup.

In [9]:
from concurrent.futures import ThreadPoolExecutor

PARALLEL_PLAN = [
    {"name": "search_products", "args": {"query": "keyboard"}},
    {"name": "search_products", "args": {"query": "webcam"}},
    {"name": "search_products", "args": {"query": "laptop"}},
]
t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(dispatch_safe, PARALLEL_PLAN))
dt = time.time() - t0
print(f"3 calls in {dt*1000:.1f} ms (parallel)")
for c, r in zip(PARALLEL_PLAN, results):
    print(f"  {c['args']['query']:<12}  hits={r['count']}")

3 calls in 1.2 ms (parallel)
  keyboard      hits=1
  webcam        hits=1
  laptop        hits=1


## 9. A lightweight observability log
Every dispatch is appended to a structured log — the data a production dashboard consumes.

In [10]:
TOOL_LOG = []

def dispatch_logged(call, auth=None):
    t0 = time.time()
    result = dispatch_safe(call, human_auth_token=auth)
    TOOL_LOG.append({
        "t":        time.time(),
        "tool":     call.get("name") if isinstance(call, dict) else "<bad>",
        "status":   "error" if "error" in result else "ok",
        "latency_ms": int((time.time() - t0) * 1000),
    })
    return result

for t in FAKE_OUTPUTS:
    parsed = extract_json(t)
    if parsed is not None:
        dispatch_logged(parsed, auth="APPROVED")

print("log rows:", len(TOOL_LOG))
for row in TOOL_LOG[:5]:
    print(" ", row)

log rows: 6
  {'t': 1776769840.380449, 'tool': 'search_products', 'status': 'ok', 'latency_ms': 0}
  {'t': 1776769840.380883, 'tool': 'send_email', 'status': 'ok', 'latency_ms': 0}
  {'t': 1776769840.380927, 'tool': 'search_products', 'status': 'ok', 'latency_ms': 0}
  {'t': 1776769840.380973, 'tool': 'search_products', 'status': 'error', 'latency_ms': 0}
  {'t': 1776769840.3813, 'tool': 'delete_order', 'status': 'error', 'latency_ms': 0}


## 10. Quick checks

In [11]:
# Every schema has the expected top-level keys
assert all({"name", "description", "parameters"} <= set(s) for s in TOOL_SCHEMAS)
# Parser recovers clean, fenced, and single-quoted forms
assert extract_json('{"a": 1}')                            == {"a": 1}
assert extract_json("```json\n{\"a\":1}\n```")              == {"a": 1}
assert extract_json("blah {'a': 1} blah")                   == {"a": 1}
assert extract_json("no object here")                        is None
# Validation rejects bad args
assert dispatch({"name": "search_products",
                 "args": {"query": "x", "limit": 999}})["error"] == "validation"
# Destructive tool requires auth
unauth = dispatch_safe({"name": "delete_order",
                        "args": {"order_id": 1, "reason": "fraud", "confirm": True}})
assert unauth["error"] == "needs_human_approval"
# Retry mechanism actually runs and succeeds on the faulty limit case
rr = run_with_retry({"name": "search_products", "args": {"query": "laptop", "limit": 999}})
assert "result" in rr and rr["attempts"] == 2
# Parallel dispatch returns a result per request
assert len(results) == len(PARALLEL_PLAN)
print("OK — schemas, parser, validator, safety gate, retry, and parallel all behave.")

OK — schemas, parser, validator, safety gate, retry, and parallel all behave.


## Reflection questions

1. Why do we return `error_type` strings like `"validation"` or `"tool_failed"` rather than raising Python exceptions? What does the LLM gain?
2. Our retry heuristic (`regenerate`) is dumb. In a real agent, what information from `TOOL_LOG` would help the LLM fix the same error next time, without a retry?
3. Destructive actions need human approval. How would you split tools into *read-only* and *write* tiers if you had 50 tools?
4. Section 8 ran three identical shape calls. What changes if one of them depends on another's result? Which agent pattern from lesson 8-1 handles that?

## References
- Source theory: [`8-2-tool-calling.mdx`](../../llm-quest-theory/level-8/8-2-tool-calling.mdx)
- Next: [`8-3-memory`](8-3-memory.ipynb)